In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("CareFlow - Event Log Normalization")
print("Libraries imported successfully")

CareFlow - Event Log Normalization
Libraries imported successfully


#### Load the original dataset

In [2]:
df = pd.read_csv("HospitalOPD-noname.csv")

# Remove completely empty columns
df = df.dropna(axis=1, how="all")

print("Dataset loaded successfully")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded successfully
Rows: 62422
Columns: 24


#### Identify the required columns

In [3]:
case_col = "appointment_id"

timestamp_columns = {
    "appointment_start_datetime": "Appointment",
    "prescription_start_datetime": "Prescription",
    "lab_start_datetime": "Lab",
    "imaging_start_datetime": "Imaging",
    "pharmacy_start_datetime": "Pharmacy"
}

print("Case ID column:", case_col)
print("Timestamp columns:", list(timestamp_columns.keys()))

Case ID column: appointment_id
Timestamp columns: ['appointment_start_datetime', 'prescription_start_datetime', 'lab_start_datetime', 'imaging_start_datetime', 'pharmacy_start_datetime']


#### Create the standardized event log

In [4]:
event_frames = []

for timestamp_col, activity_name in timestamp_columns.items():

    temp = df[[case_col, timestamp_col]].copy()

    temp.columns = ["Case_ID", "Timestamp"]

    temp["Activity_Name"] = activity_name

    event_frames.append(temp)

event_log = pd.concat(event_frames, ignore_index=True)

event_log["Timestamp"] = pd.to_datetime(
    event_log["Timestamp"], errors="coerce"
)

event_log = event_log.dropna(
    subset=["Case_ID", "Timestamp"]
)

event_log = event_log.drop_duplicates()

event_log = event_log.sort_values(
    ["Case_ID", "Timestamp", "Activity_Name"]
).reset_index(drop=True)

print("Event log created successfully")
display(event_log.head())

Event log created successfully


,Case_ID,Timestamp,Activity_Name
0,317,2024-04-25 11:12:00,Appointment
1,317,2024-04-26 05:15:00,Prescription
2,756,2024-04-27 12:32:00,Appointment
3,756,2025-08-23 14:35:00,Prescription
4,1694,2024-04-29 13:37:00,Appointment


#### Validate the event log

In [5]:
print("Total events:", len(event_log))
print("Unique cases:", event_log["Case_ID"].nunique())

print("\nMissing values:")
print(event_log.isnull().sum())

print("\nActivity counts:")
print(event_log["Activity_Name"].value_counts())

print("\nColumn names:")
print(event_log.columns.tolist())

Total events: 112517
Unique cases: 49912

Missing values:
Case_ID          0
Timestamp        0
Activity_Name    0
dtype: int64

Activity counts:
Activity_Name
Prescription    50536
Appointment     49912
Lab              6539
Imaging          5418
Pharmacy          112
Name: count, dtype: int64

Column names:
['Case_ID', 'Timestamp', 'Activity_Name']


#### Check chronological ordering

In [6]:
chronological_check = (
    event_log.groupby("Case_ID")["Timestamp"]
    .apply(lambda x: x.is_monotonic_increasing)
)

print("Cases in chronological order:",
      chronological_check.sum())

print("Cases with ordering problems:",
      (~chronological_check).sum())

Cases in chronological order: 49912
Cases with ordering problems: 0


#### Create an aggregate quality summary

In [7]:
quality_summary = pd.DataFrame({
    "Metric": [
        "Total Events",
        "Unique Cases",
        "Unique Activities",
        "Missing Case IDs",
        "Missing Timestamps",
        "Duplicate Events"
    ],
    "Value": [
        len(event_log),
        event_log["Case_ID"].nunique(),
        event_log["Activity_Name"].nunique(),
        event_log["Case_ID"].isna().sum(),
        event_log["Timestamp"].isna().sum(),
        event_log.duplicated().sum()
    ]
})

display(quality_summary)

,Metric,Value
0,Total Events,112517
1,Unique Cases,49912
2,Unique Activities,5
3,Missing Case IDs,0
4,Missing Timestamps,0
5,Duplicate Events,0


#### Save the summary

In [8]:
quality_summary.to_csv(
    "careflow_event_log_quality_summary.csv",
    index=False
)

print("Quality summary saved successfully")

Quality summary saved successfully
